# BalanceAI -- Colab GPU setup for HuatuoGPT-o1-8B (Ollama)

Reconnect procedure, after this notebook exists, is **2 real actions**:

1. Open this notebook in Colab (GPU runtime), **Runtime -> Run all**.
2. Copy the `TUNNEL URL` printed by the last cell, then on the local machine run:
   ```
   scripts/reconnect_gpu_backend.sh <tunnel-url>
   ```

What this notebook automates (previously done by hand, every single reconnect):
install zstd, install ollama, start `ollama serve`, pull the ~5.7GB HuatuoGPT
model, install + start a cloudflared tunnel, print the URL.

What's now cached across Colab VM recycles: cell 2 mounts Google Drive, and
`colab_setup.sh` points Ollama's model store (`OLLAMA_MODELS`) at a folder on
Drive instead of the ephemeral local disk. The 5.7GB model pull then only
ever happens once -- every later run, even on a fully wiped/reassigned VM,
finds the model already on Drive and skips the pull.

If the `BRANCH` variable below goes stale (branch renamed/merged), edit it
to whatever branch currently holds `ml_training/colab_setup.sh` in
https://github.com/riocorn/balanceai.

In [ ]:
# Cell 1 -- confirm a GPU runtime is attached (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
# Cell 2 -- mount Google Drive. This is what makes the HuatuoGPT model pull
# a one-time cost instead of a cost paid on every reconnect: colab_setup.sh
# (next cell) checks for /content/drive/MyDrive and, if present, stores
# pulled models there via OLLAMA_MODELS instead of local (ephemeral) disk.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 -- fetch the one real setup script from the repo and run it.
# Idempotent: safe to re-run this cell alone if the tunnel/server died mid-session.
BRANCH = "gpu-symptom-embedding-training"
!curl -fsSL -o colab_setup.sh https://raw.githubusercontent.com/riocorn/balanceai/{BRANCH}/ml_training/colab_setup.sh
!chmod +x colab_setup.sh
!bash colab_setup.sh

In [ ]:
# Cell 4 -- re-print the tunnel URL on its own (handy if you scrolled past cell 3's output).
!grep -o 'https://[a-zA-Z0-9-]*\.trycloudflare\.com' /content/balanceai_setup_logs/cloudflared.log | tail -1